# RSP Setup Check

Use this notebook first on Rubin Science Platform. It verifies the local checkout, persistent storage paths, package imports, and a tiny LSST-only ANTARES probe before any long backfill.

In [6]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    candidate = Path.home() / 'notebooks' / 'ANTARES_Analysis'
    if (candidate / 'src').exists():
        PROJECT_ROOT = candidate

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root : {PROJECT_ROOT}')
print(f'HOME         : {os.getenv("HOME")}')
print(f'USER         : {os.getenv("USER") or os.getenv("JUPYTERHUB_USER")}')
print(f'SCRATCH_DIR  : {os.getenv("SCRATCH_DIR", "not set")}')
print(f'Project area : /project/{os.getenv("USER") or os.getenv("JUPYTERHUB_USER") or "unknown_user"}')

Project root : /home/mdarim/notebooks/ANTARES_Analysis
HOME         : /home/mdarim
USER         : mdarim
SCRATCH_DIR  : /deleted-sundays/mdarim
Project area : /project/mdarim


In [7]:
import os
os.environ["ANTARES_DATA_ROOT"] = "/home/ivezic/AntaresAlerts/ANTARES_Analysis_Data"


In [8]:
import pandas as pd
import pyarrow
from antares_client.search import search as antares_search

from src import config, history, query

print('Imports complete.')
print(f'pandas  : {pd.__version__}')
print(f'pyarrow : {pyarrow.__version__}')
config.print_config_summary()

Imports complete.
pandas  : 2.3.1
pyarrow : 17.0.0
Configuration
  Last Night: MJD 61190.0 - 61191.0  (1.0 days)  [OK]
  Cumulative LSST History: MJD 61095.0 - 61190.0  (95.0 days)  [OK]
  Samples per range : 5000
  Survey mode       : lsst
  LSST-only filter  : ON
  LSST history start: MJD 61095.0
  Tag filter        : none (all alerts)
  Random seed       : 42
  Realtime night    : ON
  ANTARES lookback  : 1 day(s)
  Populated search  : ON
  Search depth      : 5 day(s)
  Chunked ingest    : ON
  Chunk start size  : 1 day(s)
  Chunk min size    : 30 sec
  Chunk split at    : 9,500/10,000 loci
  Parallel shards   : 3
  History backfill  : OFF
  History data root : /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data
  History data set  : lsst_only
  History target    : 100,000 loci/night
  History LC fetch  : ON
  Use stored history: ON

  Ranges are NON-overlapping  (MJD2_MAX=61190.0, MJD1_MIN=61190.0)


In [9]:
DATA_ROOT = Path(config.HISTORY_DATA_ROOT)
LSST_DATA_ROOT = history.survey_data_root(DATA_ROOT)
SCRATCH_ROOT = Path(os.getenv('SCRATCH_DIR', f'/scratch/{os.getenv("USER") or os.getenv("JUPYTERHUB_USER") or "unknown_user"}')) / 'ANTARES_Analysis'

for path in [
    LSST_DATA_ROOT / 'nightly',
    LSST_DATA_ROOT / 'cumulative',
    DATA_ROOT / 'data' / 'ztf_archive_do_not_use_for_lsst',
    DATA_ROOT / 'figures',
    DATA_ROOT / 'logs',
    SCRATCH_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)
    print(f'OK: {path}')

print(f'Persistent LSST store: {LSST_DATA_ROOT}')
print(f'Temporary cache root : {SCRATCH_ROOT}')


OK: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/lsst_only/nightly
OK: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/lsst_only/cumulative
OK: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/ztf_archive_do_not_use_for_lsst
OK: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/figures
OK: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/logs
OK: /deleted-sundays/mdarim/ANTARES_Analysis
Persistent LSST store: /home/ivezic/AntaresAlerts/ANTARES_Analysis_Data/data/lsst_only
Temporary cache root : /deleted-sundays/mdarim/ANTARES_Analysis


In [10]:
probe = query.query_range(
    label='RSP LSST-only probe',
    mjd_min=config.LSST_HISTORY_START_MJD,
    mjd_max=config.LSST_HISTORY_START_MJD + 1,
    n_samples=5,
    tag=config.QUERY_TAG,
    seed=None,
    verbose=True,
    lsst_only=True,
)

counts = query.lsst_identifier_counts(probe)
print(counts)
if not probe.empty:
    assert counts['lsst_identifier_count'] == len(probe), 'Probe returned non-LSST loci.'
    display_cols = [col for col in ['locus_id', 'ra', 'dec', 'newest_alert_observation_time', 'survey', 'ztf_object_id'] if col in probe.columns]
    display(probe[display_cols].head())
else:
    print('Probe returned 0 rows. Try a later MJD window before running backfill.')

  Querying 'RSP LSST-only probe'  MJD [61095.0, 61096.0]  n=5  [newest-first; LSST-only] ... retrieved 5 loci.
{'lsst_dia_count': 5, 'lsst_ss_count': 0, 'lsst_identifier_count': 5, 'ztf_object_id_count': 0}


,locus_id,ra,dec,newest_alert_observation_time,survey
0,ANT2026edokji8jctuy,187.658300,6.339556,61095.328917,"{'ztf': {'id': [], 'rcid': [], 'field': [], 's..."
1,ANT2026tx39f3zqj1p7,187.759810,6.094286,61095.328917,"{'ztf': {'id': [], 'rcid': [], 'field': [], 's..."
2,ANT2026fldi75atsesa,188.308188,6.449389,61095.328917,"{'ztf': {'id': [], 'rcid': [], 'field': [], 's..."
3,ANT2026tbfmwuwzytu2,188.511326,6.553377,61095.328917,"{'ztf': {'id': [], 'rcid': [], 'field': [], 's..."
4,ANT20262bzfhrl3z5cf,188.611184,6.613667,61095.328917,"{'ztf': {'id': [], 'rcid': [], 'field': [], 's..."
